# Unified Lab 1: The Symbolic Architect

## The Scenario
You have been hired by *Aegis Medical Systems* to build the logic core of their new automated robotic clinic.

Unlike modern deep learning models that generate probabilistic guesses based on statistics, Aegis requires absolute, mathematically provable logic. You will build a mini-Expert System (inspired by the famous MYCIN system) to diagnose patients, and then program the physical logic for a medical-delivery robot using STRIPS.

## Milestone 1: Set Up

Run this cell first.

In [ ]:
# MILESTONE 1 SETUP - RUN THIS CELL FIRST

# The Knowledge Base: A set of IF-THEN production rules.
# Each rule has a set of 'preconditions' that must be met, and a 'conclusion' to add to memory.
knowledge_base = [
    {"preconditions": {"Fever", "Cough"}, "conclusion": "Respiratory Infection"},
    {"preconditions": {"Respiratory Infection", "Shortness of Breath"}, "conclusion": "Pneumonia"},
    {"preconditions": {"Fever", "Rash", "Neck Stiffness"}, "conclusion": "Meningitis"},
    {"preconditions": {"Respiratory Infection", "Muscle Ache"}, "conclusion": "Flu"},
    {"preconditions": {"Flu", "High Risk Patient"}, "conclusion": "Prescribe Antivirals"}
]

# The Working Memory: The initial facts (symptoms) we know about Patient A
patient_a_symptoms = {"Fever", "Cough", "Muscle Ache", "High Risk Patient"}

print("Aegis Medical Knowledge Base Loaded.")
print(f"Total Rules: {len(knowledge_base)}")
print(f"Patient A Initial Facts: {patient_a_symptoms}")

## Milestone 1: The Inference Engine (Forward Chaining)
**Objective:** The clinic needs a triage system that takes in known patient facts (symptoms) and automatically deduces any possible diagnoses using **Forward Chaining** (Data-Driven Reasoning).

You will build a pure Python Inference Engine that executes the **Match-Select-Act** cycle. Given a Knowledge Base of medical rules and a patient's initial symptoms, your engine must continuously fire rules until no new medical facts can be deduced.

### The Architect's Blueprint
*Write a Python function `forward_chaining(rules, initial_facts)` that implements the Match-Select-Act cycle.*

* **1. Initialize Working Memory:** Create a new Python `set` called `working_memory` using the `initial_facts`.
* **2. The Inference Loop:** Create a `while` loop that will keep running as long as new facts are being discovered. (Hint: Use a boolean flag like `new_fact_added = True` to control the loop. Set it to `False` at the start of each cycle).
* **3. Match & Select:** Inside the loop, iterate through every `rule` in the `rules` list.
    * Check if the rule's `preconditions` are completely contained within the `working_memory`. (Hint: Python sets have a handy `.issubset()` method).
    * Check if the rule's `conclusion` is *not* already in the `working_memory`.
* **4. Act:** If both conditions above are met, add the `conclusion` to the `working_memory`, print a message saying which rule fired (e.g., `"Rule Fired: Inferred [Flu]"`), and flip your `new_fact_added` flag back to `True` so the loop runs again.
* **5. Return:** Once the loop finishes (meaning no new rules can fire), return the final `working_memory`.

In [ ]:
def forward_chaining(rules, initial_facts):
    working_memory = set(initial_facts)
    new_fact_added = True
    while new_fact_added:
        new_fact_added = False
        for rule in rules:
            if rule["preconditions"].issubset(working_memory) and rule["conclusion"] not in working_memory:
                working_memory.add(rule["conclusion"])
                print(f"Rule Fired: Inferred [{rule['conclusion']}]")
                new_fact_added = True
    return working_memory

### The Test Harness
*Use this exact block of code; your AI's generated code MUST pass these assertions.*

In [ ]:
# MILESTONE 1 TEST HARNESS - DO NOT MODIFY

print("--- Starting Aegis Inference Engine ---")
final_memory = forward_chaining(knowledge_base, patient_a_symptoms)
print("\n--- Final Working Memory ---")
print(final_memory)

# Assertions
assert "Respiratory Infection" in final_memory, "Failure: The engine failed to infer a basic respiratory infection."
assert "Flu" in final_memory, "Failure: The engine failed to chain 'Respiratory Infection' into 'Flu'."
assert "Prescribe Antivirals" in final_memory, "Failure: The engine failed a deep chain (Fever+Cough -> Resp. Infection -> Flu -> Prescribe Antivirals)."
assert "Pneumonia" not in final_memory, "Failure: The engine hallucinated Pneumonia without the Shortness of Breath symptom!"
assert "Meningitis" not in final_memory, "Failure: The engine fired a rule without all preconditions being met."

print("\nSUCCESS! The Forward Chaining Engine is mathematically sound.")

### Architect's Audit (Milestone 1)

1. **The Match-Select-Act Cycle:** Looking at the print statements from your code, explain how the engine used a fact it discovered in Cycle 1 to trigger a completely new rule in Cycle 2. Why is a `while` loop mathematically necessary for Forward Chaining to work?
2. **The "Rule Explosion" Limitation:** In Lecture 1.5, we discussed the limits of Expert Systems. Our Aegis Knowledge Base only has 5 rules, making it extremely fast. Imagine a real hospital Expert System with 50,000 overlapping rules. Based on the `for` loop you just wrote, explain why pure Forward Chaining becomes computationally expensive and incredibly difficult for human engineers to maintain as the Knowledge Base grows.

ENTER YOUR ANSWERS HERE

1.
Examining the trace shows that all three rules—Respiratory Infection, Flu, and Prescribe Antivirals—were triggered simultaneously in the same loop iteration (Cycle 1), rather than in separate passes. This happens because working_memory is updated directly as the loop processes the rule list: since the knowledge_base lists Respiratory Infection before Flu before Prescribe Antivirals, the relevant facts for subsequent rules are already added during the same pass. In Cycle 2, the system rechecks all five rules, finds none newly satisfied, and then exits. Mathematically, this process is a fixed-point iteration in which each cycle applies a monotonic function to working_memory. The loop ensures convergence to a complete, order-independent set of facts, regardless of the order of rules.

2.
The inner for loop in forward_chaining examines each rule in rules on every iteration of the while loop, using issubset() to compare each rule with working_memory, even if the rule isn't relevant to the current patient. With only 5 rules, this process is instantaneous, but with 50,000 overlapping rules, each cycle requires a full linear scan of the knowledge base—and the total number of cycles depends on the length of the longest inference chain triggered by a patient's symptoms. 

Therefore, the total effort is approximately (rules) times (cycles), with most of the 50,000 checks per cycle being for rules that are not applicable to this patient. Unlike the goal-focused Backward Chaining from Milestone 2, which assesses only rules related to the hypothesis, Forward Chaining is data-driven — it cannot predict which rules are relevant beforehand and must perform a brute-force check of all rules each cycle until no new rules activate.

Beyond just runtime issues, a more significant challenge for human engineers is managing the complexity of 50,000 overlapping rules. These rules can share facts across independently written, different-time contributors. Making changes—adding or editing a rule—may silently alter which other rules are triggered. For example, a "Penicillin Allergy" rule might interact unexpectedly with an unrelated conclusion, leading to unforeseen diagnoses that are hard to trace from the new rule alone. 

## Milestone 2: Set Up

Run this cell first.

In [ ]:
# MILESTONE 2 SETUP - RUN THIS CELL FIRST

knowledge_base = [
    {"preconditions": {"Fever", "Cough"}, "conclusion": "Respiratory Infection"},
    {"preconditions": {"Respiratory Infection", "Shortness of Breath"}, "conclusion": "Pneumonia"},
    {"preconditions": {"Fever", "Rash", "Neck Stiffness"}, "conclusion": "Meningitis"},
    {"preconditions": {"Respiratory Infection", "Muscle Ache"}, "conclusion": "Flu"},
    {"preconditions": {"Flu", "High Risk Patient"}, "conclusion": "Prescribe Antivirals"}
]

# Patient B's initial symptoms
patient_b_symptoms = {"Fever", "Cough", "Muscle Ache", "High Risk Patient"}

# The Hypothesis we want to test
hypothesis = "Prescribe Antivirals"
false_hypothesis = "Meningitis"

print("Aegis Medical Knowledge Base Loaded.")
print(f"Testing Hypothesis: [{hypothesis}] on Patient B.")

## Milestone 2: The Hypothesis Tester (Backward Chaining)
**Objective:** The clinic's Lead Diagnostician steps in. They are dealing with Patient B in the ER. They do not want the system to waste compute cycles inferring every possible minor ailment. They only have one specific, high-stakes question: *"Based on the current facts, is it logically valid to Prescribe Antivirals to Patient B?"*

Instead of data-driven reasoning, you will implement **Backward Chaining** (Goal-Driven Reasoning). You will write a recursive function that starts at the ultimate goal, looks for the rules needed to prove it, and recursively traces backward to see if the patient's initial symptoms support those rules.

### The Architect's Blueprint (Student Prompt)
*Write a recursive Python function `backward_chaining(goal, rules, known_facts)`.*

* **1. The Base Case:** First, check if the `goal` is already inside `known_facts`. If it is, print a message like `"Fact confirmed: [goal]"` and immediately return `True`.
* **2. Find the Relevant Rules:** Create a `for` loop to iterate through the `rules`. We only care about rules where the `rule["conclusion"]` matches our current `goal`.
* **3. The Recursive Step:** If you find a rule that matches the goal, you must assume it *might* work.
    * Create a flag: `all_preconditions_met = True`.
    * Loop through every `precondition` inside that rule's `rule["preconditions"]`.
    * Call your `backward_chaining()` function recursively, passing the `precondition` in as the new `goal`.
    * If the recursive call returns `False` for *any* precondition, flip your flag to `all_preconditions_met = False` and `break` out of the precondition loop (because if one precondition fails, the whole rule fails).
* **4. Success:** If the precondition loop finishes and `all_preconditions_met` is still `True`, print `"Rule proven for: [goal]"` and return `True`.
* **5. The Failure State:** If the outer rule loop finishes and you haven't returned `True`, it means no rules could prove the goal. Return `False`.

In [ ]:
def backward_chaining(goal, rules, known_facts):
    if goal in known_facts:
        print(f"Fact confirmed: [{goal}]")
        return True
    for rule in rules:
        if rule["conclusion"] == goal:
            all_preconditions_met = True
            for precondition in rule["preconditions"]:
                if not backward_chaining(precondition, rules, known_facts):
                    all_preconditions_met = False
                    break
            if all_preconditions_met:
                print(f"Rule proven for: [{goal}]")
                return True
    return False

### The Test Harness
*Run this code after writing your blueprint to verify your recursive Inference Engine works.*

In [ ]:
# MILESTONE 2 TEST HARNESS - DO NOT MODIFY

print("--- Testing True Hypothesis ---")
result_true = backward_chaining(hypothesis, knowledge_base, patient_b_symptoms)

print("\n--- Testing False Hypothesis ---")
result_false = backward_chaining(false_hypothesis, knowledge_base, patient_b_symptoms)

# Assertions
assert result_true == True, f"Failure: The engine failed to prove {hypothesis}, even though the patient has all base symptoms."
assert result_false == False, f"Failure: The engine hallucinated a proof for {false_hypothesis}!"

print("\nSUCCESS! The Backward Chaining Engine successfully tested the hypotheses.")

### Architect's Audit (Milestone 2)

1. **The Recursive Trace:** Look at your print statements for the True Hypothesis. Write down the exact order of the "sub-goals" the engine tested to eventually prove "Prescribe Antivirals." How does this order visually demonstrate the concept of Goal-Driven Reasoning?
2. **Efficiency at Scale:** In Milestone 1, we established that Forward Chaining checks every rule against every fact, causing "Rule Explosion." Using the definitions from Lecture 1.3, explain to the Lead Diagnostician why your new Backward Chaining engine is vastly more efficient for answering their specific question about Antivirals.

ENTER YOUR ANSWERS HERE

1.
Starting from the top-level goal "Prescribe Antivirals," the engine proceeds to evaluate its two preconditions: "High Risk Patient" and "Flu." Since "High Risk Patient" is already known, it is confirmed immediately (line 1). For "Flu," which isn't yet known, the engine explores its preconditions — "Respiratory Infection" and "Muscle Ache." "Respiratory Infection" is also unknown, prompting the engine to delve further into its preconditions, "Fever" and "Cough," both of which are confirmed facts (lines 2–3). Once these leaf facts are established, "Respiratory Infection" is confirmed (line 4), enabling the engine to verify "Muscle Ache," the other precondition of "Flu" (line 5), thereby confirming "Flu" itself (line 6). With both preconditions of "Prescribe Antivirals" verified, the top-level goal is finally proven (line 7).

This order illustrates Goal-Driven Reasoning by tracing backward from the query (Prescribe Antivirals), breaking it into smaller sub-goals, and reaching known facts. The "proven" messages then ascend back up the call stack in reverse (Respiratory Infection → Flu → Prescribe Antivirals), with each step only confirmed once its preconditions are met. Importantly, the engine ignores unrelated rules, like Pneumonia or Meningitis, focusing solely on the dependency chain relevant to the specific question, rather than examining the entire knowledge base.

2.
Forward chaining has a stopping condition—it ends when the goal is achieved or no further rules fire. However, it doesn't consider what the goal is when selecting rules. Each cycle, it advances from data to conclusions based on current facts, regardless of whether those conclusions relate to the lead diagnostician's question. With 50,000 rules, performance issues are not in rescanning the rule base—which production systems optimize using Rete—but in the derivation process itself, which generates conclusions about conditions like pneumonia and meningitis driven by the facts, not the user's needs.

In contrast, backward chaining flips this approach. The goal becomes the criterion for selecting rules, focusing only on those whose conclusions match what is being proved. For example, in our trace, only three rules—Prescribe Antivirals, Flu, Respiratory Infection—are examined, out of five, because the chain for antivirals doesn't involve pneumonia or meningitis.
With that in mind, the Diagnostician pays only for the rules that connect their question to the known facts, not for the entire database. This trade-off works both ways. Backward chaining needs a predetermined goal, making it unsuitable for ongoing monitoring. Conversely, forward chaining's ability to derive all conclusions is ideal in such cases. The key architectural consideration isn't which method is superior, but whether the system is designed to answer specific queries or to monitor a continuous stream.

## Milestone 3: Set Up

Run this cell first.

In [ ]:
# MILESTONE 3 SETUP - RUN THIS CELL FIRST

# The Initial State of the physical world
world_state = {"RobotAt(Hallway)", "MedicineAt(Pharmacy)", "PatientAt(RoomB)"}

# The STRIPS Action Definitions
# Each action has preconditions that must be true, effects to ADD, and effects to DELETE
medbot_actions = {
    "MoveToPharmacy": {
        "preconditions": {"RobotAt(Hallway)"},
        "add": {"RobotAt(Pharmacy)"},
        "delete": {"RobotAt(Hallway)"}
    },
    "FetchMedicine": {
        "preconditions": {"RobotAt(Pharmacy)", "MedicineAt(Pharmacy)"},
        "add": {"RobotHas(Medicine)"},
        "delete": {"MedicineAt(Pharmacy)"}
    },
    "MoveToRoomB": {
        "preconditions": {"RobotAt(Pharmacy)", "RobotHas(Medicine)"},
        "add": {"RobotAt(RoomB)"},
        "delete": {"RobotAt(Pharmacy)"}
    },
    "DeliverMedicine": {
        "preconditions": {"RobotAt(RoomB)", "RobotHas(Medicine)", "PatientAt(RoomB)"},
        "add": {"PatientHas(Medicine)"},
        "delete": {"RobotHas(Medicine)"}
    }
}

print("Aegis MedBot Initialized.")
print(f"Current World State: {world_state}")

## Milestone 3: The MedBot (STRIPS Planning)
**Objective:** The Inference Engine has successfully diagnosed Patient B and concluded: `"Prescribe Antivirals"`. Now, Aegis Medical Systems must physically deliver the medication.

You step away from logical inference and move into physical planning. You will program the core state-transition logic for the clinic's automated MedBot using **STRIPS** (Stanford Research Institute Problem Solver). You will write a function that takes a current physical state, checks the preconditions of a physical action, and mathematically updates the world state by applying the `Delete` list and then the `Add` list.

### The Architect's Blueprint (Student Prompt)
*Write a Python function `execute_action(current_state, action_definition)` that applies a STRIPS action to the world, and then manually execute a hard-coded sequence of actions to deliver the medicine.*

* **1. The Action Engine:** Write the `execute_action` function.
    * First, check if the action's `preconditions` are a subset of the `current_state`.
    * If they are NOT met, `print` an error message (e.g., `"Action Failed: Preconditions not met."`) and return the `current_state` unchanged.
    * If they ARE met, you must update the state. *Crucial STRIPS Math:* First, remove everything in the `delete` list from the `current_state`. Then, add everything in the `add` list to the `current_state`. (Hint: Python sets have `.difference()` and `.union()` methods, or you can use the `-` and `|` operators).
    * Return the new state.
* **2. The Execution Plan:** We know the goal is to get the medicine to the patient. Define a Python list called `plan` containing the string names of the actions in the correct logical order: Move to Pharmacy -> Fetch -> Move to Room B -> Deliver.
* **3. Run the Robot:** Create a `for` loop that iterates through your `plan`.
    * Inside the loop, extract the specific `action_definition` from the `medbot_actions` dictionary using the action's name.
    * Call `world_state = execute_action(world_state, action_definition)`.
    * Print the `world_state` after every action so you can see the robot moving through the world!

In [ ]:
# ENTER YOUR CODE HERE

### The Test Harness
*Run this code after writing your blueprint to verify the MedBot successfully altered the physical world state.*

In [ ]:
# MILESTONE 3 TEST HARNESS - DO NOT MODIFY

assert "PatientHas(Medicine)" in world_state, "Failure: The patient did not receive the medicine."
assert "RobotAt(RoomB)" in world_state, "Failure: The robot is not in the correct final location."
assert "RobotAt(Hallway)" not in world_state, "Failure: The Delete list failed. The robot is magically in two places at once!"
assert "RobotHas(Medicine)" not in world_state, "Failure: The robot didn't give the medicine to the patient, it kept it!"

print("\nSUCCESS! The MedBot successfully completed the physical STRIPS plan.")

### Architect's Audit (Milestone 3)

1. **The STRIPS State Update:** Look at your print statements. Explain exactly how the `Delete` list and `Add` list worked together during the `MoveToPharmacy` action to accurately reflect physical movement in the `world_state` set. Why is the `Delete` list logically mandatory for physical actions?
2. **The Limits of Symbolic AI Robotics:** Based on Lecture 1.6, STRIPS makes a massive, often unrealistic assumption about the physical world. Imagine the MedBot is executing `MoveToRoomB`, but a doctor accidentally left a medical cart blocking the hallway, or the robot's battery dies mid-transit. Explain why this purely deterministic STRIPS model would catastrophically fail in a noisy, real-world hospital.

ENTER YOUR ANSWERS HERE

## The Summary Audit
To complete the Lab, you must submit your final Colab Notebook containing your code along with brief 3-4 sentence answers to the following questions synthesizing the module:

1. **The Power of Explicit Logic:** In Milestones 1 and 2, you built an Inference Engine. Explain why a hospital might legally or ethically prefer this rigid, rule-based "Expert System" over a modern, Deep Learning neural network (like ChatGPT) when diagnosing a patient's symptoms.
2. **The AI Winter:** Despite their logical perfection, Expert Systems faded from prominence in the late 1980s, leading to an "AI Winter." Based on the lectures, explain the core limitations of hand-coded Rule-Based Systems and STRIPS Planning when confronted with the massive scale and ambiguity of real-world data.
3. **The Hybrid Future:** We do not use pure symbolic AI much today, but we are starting to combine it with Machine Learning. How might a modern Hybrid System combine the statistical "guessing" power of a Deep Neural Network with the rigid, rules-based logic of the Inference Engine you built today?

ENTER YOUR ANSWERS HERE